# 01 — Compréhension des données

**Objectif** : décrire `data/raw/donn_prix_vente_reqst.csv` sans le modifier.

Le fichier provient du [Registre foncier du Québec](https://www.donneesquebec.ca/recherche/dataset/statistiques-du-registre-foncier-du-quebec-sur-le-marche-immobilier)
(MRNF). Chaque ligne est le **nombre de ventes inscrites** pour un mois, une
région administrative et une plage de prix.

Aucune correction n'est appliquée ici : le nettoyage appartient au notebook `02`.

In [14]:
import importlib
import sys
from pathlib import Path

racine = Path.cwd().resolve()
if not (racine / "src").is_dir():
    racine = racine.parent
sys.path.insert(0, str(racine))

import src
import src.referentiels

importlib.reload(src)
importlib.reload(src.referentiels)

import pandas as pd

from src import VENTES_BRUTES_CSV
from src.referentiels import PLAGES_PRIX, REGIONS_ADMINISTRATIVES

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

## Chargement brut

Tout est lu en texte (`dtype=str`) pour observer les valeurs telles qu'elles
figurent dans le fichier, avant toute conversion implicite par pandas.

In [ ]:
brut = pd.read_csv(VENTES_BRUTES_CSV, dtype=str, encoding="utf-8-sig")

print(f"{brut.shape[0]} lignes × {brut.shape[1]} colonnes")
print(f"Fichier : {VENTES_BRUTES_CSV.name}")
brut.head(10)

3262 lignes × 5 colonnes
Fichier : donn_prix_vente_reqst.csv


,DT_DEBUT_MOIS,ID_REGN_ADMIN,CD_PLAGE_PRIX,CD_NATR_ACTE_JURDQ,NB_REQST
0,2021-01-01,01,1,001,240
1,2021-01-01,01,2,001,29
2,2021-01-01,01,3,001,6
3,2021-01-01,02,1,001,298
4,2021-01-01,02,2,001,48
5,2021-01-01,02,3,001,16
6,2021-01-01,03,1,001,497
7,2021-01-01,03,2,001,295
8,2021-01-01,03,3,001,84
9,2021-01-01,04,1,001,345


## Dictionnaire des colonnes

| Colonne source | Sens | Type attendu |
| --- | --- | --- |
| `DT_DEBUT_MOIS` | Premier jour du mois d'inscription | date |
| `ID_REGN_ADMIN` | Région administrative (01 à 17, sans 10) | code |
| `CD_PLAGE_PRIX` | Plage de prix de vente (1, 2, 3) | code |
| `CD_NATR_ACTE_JURDQ` | Nature de l'acte juridique (`001` = vente) | code |
| `NB_REQST` | Nombre de ventes inscrites | entier |

Les ventes regroupent **tous les types de propriétés** (unifamiliale, plex,
condo, etc.). Un acte qui contient plusieurs ventes n'est compté qu'une fois
(premier droit de l'acte).

## Structure et types

In [16]:
brut.info()

<class 'pandas.DataFrame'>
RangeIndex: 3262 entries, 0 to 3261
Data columns (total 5 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   DT_DEBUT_MOIS       3262 non-null   str  
 1   ID_REGN_ADMIN       3262 non-null   str  
 2   CD_PLAGE_PRIX       3262 non-null   str  
 3   CD_NATR_ACTE_JURDQ  3262 non-null   str  
 4   NB_REQST            3262 non-null   str  
dtypes: str(5)
memory usage: 187.2 KB


In [17]:
pd.read_csv(VENTES_BRUTES_CSV, encoding="utf-8-sig").dtypes

DT_DEBUT_MOIS           str
ID_REGN_ADMIN         int64
CD_PLAGE_PRIX         int64
CD_NATR_ACTE_JURDQ    int64
NB_REQST              int64
dtype: object

## Couverture temporelle et géographique

In [18]:
periodes = pd.to_datetime(brut["DT_DEBUT_MOIS"], errors="coerce")
regions_obs = sorted(brut["ID_REGN_ADMIN"].dropna().unique())
regions_officielles = list(REGIONS_ADMINISTRATIVES)

print(f"Période : {periodes.min():%Y-%m} → {periodes.max():%Y-%m}")
print(f"Mois distincts : {periodes.nunique()}")
print(f"Régions observées : {regions_obs}")
print(f"Régions absentes  : {sorted(set(regions_officielles) - set(regions_obs))}")
print(
    "La région 10 (Nord-du-Québec) est volontairement ventilée dans les "
    "régions 02, 08 et 09 par le producteur."
)

Période : 2021-01 → 2026-08
Mois distincts : 68
Régions observées : ['01', '02', '03', '04', '05', '06', '07', '08', '09', '11', '12', '13', '14', '15', '16', '17']
Régions absentes  : ['10']
La région 10 (Nord-du-Québec) est volontairement ventilée dans les régions 02, 08 et 09 par le producteur.


## Valeurs manquantes et doublons

In [19]:
manquants = pd.DataFrame(
    {
        "n_manquants": brut.isna().sum(),
        "pct_manquants": (brut.isna().mean() * 100).round(2),
        "n_uniques": brut.nunique(dropna=True),
    }
)
manquants

,n_manquants,pct_manquants,n_uniques
DT_DEBUT_MOIS,0,0.0,68
ID_REGN_ADMIN,0,0.0,16
CD_PLAGE_PRIX,0,0.0,3
CD_NATR_ACTE_JURDQ,0,0.0,1
NB_REQST,0,0.0,929


In [20]:
cles = ["DT_DEBUT_MOIS", "ID_REGN_ADMIN", "CD_PLAGE_PRIX", "CD_NATR_ACTE_JURDQ"]
print(f"Lignes entièrement dupliquées : {brut.duplicated().sum()}")
print(f"Doublons sur la clé métier    : {brut.duplicated(subset=cles).sum()}")

Lignes entièrement dupliquées : 0
Doublons sur la clé métier    : 0


## Modalités et volumes

In [21]:
for colonne in brut.columns:
    valeurs = brut[colonne].dropna().unique()
    if len(valeurs) <= 30:
        print(f"\n{colonne} ({len(valeurs)} modalités)")
        for valeur in sorted(valeurs):
            print(f"  - {valeur}")


ID_REGN_ADMIN (16 modalités)
  - 01
  - 02
  - 03
  - 04
  - 05
  - 06
  - 07
  - 08
  - 09
  - 11
  - 12
  - 13
  - 14
  - 15
  - 16
  - 17

CD_PLAGE_PRIX (3 modalités)
  - 1
  - 2
  - 3

CD_NATR_ACTE_JURDQ (1 modalités)
  - 001


In [22]:
brut["NB_REQST_num"] = pd.to_numeric(brut["NB_REQST"], errors="coerce")

par_plage = (
    brut.groupby("CD_PLAGE_PRIX", as_index=False)["NB_REQST_num"]
    .sum()
    .assign(libelle=lambda d: d["CD_PLAGE_PRIX"].map(PLAGES_PRIX))
)
par_plage

,CD_PLAGE_PRIX,NB_REQST_num,libelle
0,1,364135,Moins de 250 000 $
1,2,398901,De 250 000 $ à 500 000 $
2,3,287201,Plus de 500 000 $


## Constats à reporter dans le notebook 02

- **Grain** : une ligne = un mois × une région × une plage de prix.
- **Cible** : `NB_REQST` (nombre de ventes), déjà numérique dans l'export.
- **Période** : `DT_DEBUT_MOIS` au format `YYYY-MM-01`.
- **Clé d'unicité** : les quatre colonnes de codes.
- **Région 10 absente** : à documenter, pas à imputer.
- **Plages** : `1` < 250 000 $, `2` 250–500 k$, `3` > 500 000 $.
- **Nature d'acte** : une seule modalité (`001` = vente) dans ce fichier.